<div align="right"><a href="S19-Design-of-a-Hilbert-transformer-Remez-algorithm.ipynb">Sprache wechseln zu Englisch 🇬🇧</a></div>

Die folgenden Übungen sind Teil des Buches
* K.-D. Kammeyer, K. Kroschel: "Digitale Signalverarbeitung - Filterung und Spektralanalyse", Springer Vieweg, 2022.  [[KK22]](https://doi.org/10.1007/978-3-663-09805-8)

---

Dieses Jupyter Notebook kann durch Klicken auf das Symbol rechts in [Google Colab](https://colab.research.google.com/notebooks/intro.ipynb) geöffnet werden. Am besten wird es jedoch lokal mit [Jupyter Notebook](https://jupyter.org/) angezeigt (z. B. als Teil der [Anaconda](https://www.anaconda.com/)-Distribution). <a href="https://colab.research.google.com/github/Stefan-Goe/dsp-book/blob/main/python/L19-Entwurf-eines-Hilbert-Transformators-%28Remez-Verfahren%29.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open Notebook in Google Colab"></a>

---

<!--NAVIGATION-->
<div align="center">

Zurück: [L17 Remez-Entwurf (Tschebyscheff-Approximation)](./L17-Remez-Entwurf-%28Tschebyscheff-Approximation%29.ipynb) | [Aufgabe 19](./A19-Entwurf-eines-Hilbert-Transformators-%28Remez-Verfahren%29.ipynb) | [Inhalt / Übersicht](./00-Python-Notebooks-Overview-DE.ipynb) | Weiter: [L20 Eigenschaften der Diskreten Fouriertransformation (DFT)](./L20-Diskrete-Fouriertransformation-%28DFT%29.ipynb)

</div>

---


### Aufgabe: Entwurf eines Hilbert-Transformators

In dieser Aufgabe soll ein Hilbert-Transformator mit idealer Frequenzantwort

$$
H(e^{j\Omega}) = j\,\mathrm{sgn}(\Omega)
$$

mithilfe des Remez-Algorithmus (`scipy.signal.remez()` in Python) approximiert werden.

---

#### a) 

Entwerfen Sie zwei Hilbert-Transformatoren der Ordnung 30 und 31 unter Verwendung des zusätzlichen Arguments `'hilbert'`:

```python
from scipy.signal import remez

b = remez(numtaps=m, bands=F, desired=G, type='hilbert')
```

Plotten Sie die Amplituden- und Impulsantwort sowie die Pol-Nullstellen-Diagramme. Welche Filtertypen wurden erstellt?

---

#### b)

Erstellen Sie ein Sinussignal mit einer Dauer von zwei Sekunden, einer Signalfrequenz von $50\text{Hz}$ und einer Abtastfrequenz von $1000\text{Hz}$.

Filtern Sie dieses Signal mit dem Hilbert-Transformator der Ordnung $m=30$ und plotten Sie Eingangs- und Ausgangssignal.

*Hinweis:* Die Gruppenverzögerung muss berücksichtigt werden, da das Ausgangssignal verzögert ist.

---

#### c)

Für die Hilbert-Transformation (wichtig in der Kommunikationstechnik) bietet Python eine eigene Funktion:

```python
from scipy.signal import hilbert

y = hilbert(x)
```

Der Rückgabewert `y` ist das analytische Signal zu `x`, der Realteil von `y` entspricht `x` und der Imaginärteil von `y` ist die Hilbert-Transformation von `x`.

Berechnen Sie die Hilbert-Transformation des Signals aus Aufgabe **b** und vergleichen Sie die Ergebnisse.

In [ ]:
# BEGIN SOLUTION
import numpy as np
from scipy import signal
import matplotlib.pyplot as plt
from matplotlib.patches import Circle

# Note: The FIR coefficients (b) from SciPy were negated (-b_scipy) to match 
# the sign convention and plot orientation of the original MATLAB solution.

# ##### Task a: Design the Hilbert Transformer #####

# Filter Order 30 (Type II)
order1 = 30
# Remez design. Normalized frequency bands [0.05, 0.95]
b1_scipy = signal.remez(order1 + 1, [0.05, 0.95], [1], Hz=2.0, type='hilbert')
b1 = -b1_scipy 

# Filter Order 31 (Type IV)
order2 = 31
b2_scipy = signal.remez(order2 + 1, [0.05, 1], [1], Hz=2.0, type='hilbert')
b2 = -b2_scipy

# Calculate Frequency Response
W1, H1 = signal.freqz(b1, 1, 512, whole=False)
W2, H2 = signal.freqz(b2, 1, 512, whole=False)

# Plot 1: Magnitude Response, Order 30
plt.figure(figsize=(10, 4))
plt.plot(W1 / np.pi, np.abs(H1))
plt.grid(True)
plt.xlabel('Omega/pi')
plt.ylabel('|H_H(exp(j*Omega))|')
plt.title('Magnitude Response - Filter Order: 30 Type II')

# Plot 2: Magnitude Response, Order 31
plt.figure(figsize=(10, 4))
plt.plot(W2 / np.pi, np.abs(H2))
plt.grid(True)
plt.xlabel('Omega/pi')
plt.ylabel('|H_H(exp(j*Omega))|')
plt.title('Magnitude Response - Filter Order: 31 Type IV')

# Plot 3: Impulse Response, Order 30
plt.figure(figsize=(10, 4))
plt.stem(np.arange(order1 + 1), b1, basefmt="r-", markerfmt="o", linefmt="C0-")
plt.axis([0, order1, -0.8, 0.8])
plt.grid(True)
plt.xlabel('k')
plt.ylabel('Impulsantwort')
plt.title('Impulse Response - Filter Order: 30 Type II')

# Plot 4: Impulse Response, Order 31
plt.figure(figsize=(10, 4))
plt.stem(np.arange(order2 + 1), b2, basefmt="r-", markerfmt="o", linefmt="C0-")
plt.axis([0, order2, -0.8, 0.8])
plt.grid(True)
plt.xlabel('k')
plt.ylabel('Impulsantwort')
plt.title('Impulse Response - Filter Order: 31 Type IV')

# ##### Task b: Filter Output Signal #####

# Coefficients for filtering (Order 30)
b_scipy = signal.remez(30 + 1, [0.05, 0.95], [1], Hz=2.0, type='hilbert')
b = -b_scipy 
Fs = 1000  
t = np.arange(0, 2, 1/Fs) 
x = np.sin(2 * np.pi * 50 * t) 
xh = signal.lfilter(b, 1, x)  

# Plot 5: Pole-Zero Diagram (zplane)
z, p, k = signal.tf2zpk(b, [1])

plt.figure(figsize=(6, 6))
ax = plt.gca()
# Zeros
ax.plot(np.real(z), np.imag(z), 'o', fillstyle='none', markersize=8, color='C0', label='Zeros (Z)')
# Poles (All lie at z=0 for FIR filter. Manual large marker for visibility.)
ax.plot(0, 0, 'x', markersize=12, markeredgewidth=2, color='blue', label='Poles (P)')

# Unit circle
unit_circle = Circle((0, 0), 1, color='gray', linestyle='--', fill=False)
ax.add_artist(unit_circle)
ax.axvline(0, color='gray')
ax.axhline(0, color='gray')
ax.axis('equal')
ax.set_xlim([-1.2, 1.2])
ax.set_ylim([-1.2, 1.2])
ax.set_xlabel('Realteil')
ax.set_ylabel('Imaginärteil')
ax.set_title('Pole-Zero Diagram')
ax.grid(True)
ax.legend()


delay = 15
xd = np.concatenate((np.zeros(delay), x[:len(x) - delay]))

# Plot 6: Input vs. Filtered Output
num_samples = 50
plt.figure(figsize=(10, 4))
plt.plot(t[:num_samples], xd[:num_samples], label='- Input Sig.')
plt.plot(t[:num_samples], xh[:num_samples], '--', label='-- Output Sig.')
plt.grid(True)
plt.xlabel('Time (sec)')
plt.ylabel('Amplitude')
plt.title(' - Input Sig.   -- Output Sig.')
plt.legend(loc='upper right')

# ------------------------------------------------------------------------------
# ##### Task c: Calculate Hilbert Transform #####

Fs = 1000
t = np.arange(0, 2, 1/Fs)
x = np.sin(2 * np.pi * 50 * t)
y = signal.hilbert(x) # Analytic signal y = x + j*xh

# Plot 7: Original vs. Hilbert Transform
num_samples = 50
plt.figure(figsize=(10, 4))
plt.plot(t[:num_samples], np.real(y[:num_samples]), label='- Original') 
plt.plot(t[:num_samples], np.imag(y[:num_samples]), '--', label='-- Hilbert Transform.')
plt.grid(True)
plt.xlabel('Time (sec)')
plt.ylabel('Amplitude')
plt.title('-- Hilbert-Transform.   - Original')
plt.legend(loc='upper right')

plt.show() 
# END SOLUTION

# Lösung: Entwurf eines Hilbert-Transformators

### a) Filterentwurf und Impuls-/Amplitudenantwort

Wenn wir den optionalen Parameter[^1] berücksichtigen, erhalten wir einen Filter vom Typ II für die Filterordnung $m=30$ und einen Filter vom Typ IV für die Filterordnung $m=31$. Die folgenden Abbildungen zeigen die Amplituden- und Impulsantworten der Filterentwürfe.

<img src="./pngs/ldsv5a05_a1.png" width="250">
<img src="./pngs/ldsv5a05_a3.png" width="250">
<img src="./pngs/ldsv5a05_a2.png" width="250">
<img src="./pngs/ldsv5a05_a4.png" width="250">

Es zeigt sich außerdem, dass die Annäherungsqualität bei ungerader Filterordnung höher ist, da die Nullstelle bei $\Omega = \pi$ fehlt.

Um einen Hilbert-Transformator zur Erzeugung analytischer Signale zu verwenden, müssen das Ausgangssignal (Imaginärteil) und das Eingangssignal (Realteil) gleich verzögert und kombiniert werden. Das bedeutet, dass eine gerade Gruppendelay existieren sollte. In einem nichtrekursiven, linearen System trägt jede Nullstelle $-0.5$ zur Gruppendelay bei. Daher müsste für einen Hilbert-Transformator eine gerade Filterordnung existieren. Dies widerspricht jedoch dem Ziel, eine hohe Annäherungsqualität zu erzielen.

Zur Verdeutlichung der Nullstellenposition zeigt die folgende Abbildung die Nullstellenverteilung des Filters der Ordnung $m=30$.

<img src="./pngs/ldsv5a05_a5.png" width="250">

Es gibt $15$ Nullstellen, die entweder auf dem Einheitskreis liegen oder als Paar spiegelbildlich auf dem Einheitskreis angeordnet sind, da das System linear und nichtrekursiv ist.

[^1]: Wenn dieser Parameter nicht festgelegt ist, wird der Filter vom Typ I/III erzeugt. Das Design ist daher nicht impulse-invariant, siehe Aufgabe 5.4.a.  
*Hinweis:* In Python kann der entsprechende Filter mit `scipy.signal.remez()` und dem Argument `type='hilbert'` erzeugt werden.

---

### b) Gruppendelay und Signalüberprüfung

Berücksichtigt man die Gruppendelay, so ist das Signal bei einer Filterordnung von $m=30$ um $15$ Samples verzögert, entsprechend Aufgabe a).  

Die folgende Abbildung überprüft die Beziehung  

$$
-\cos(x) = \mathcal{H}\{\sin(x)\}
$$  

für eine Sinusreihe mit der Hilbert-Transformation $\mathcal{H}\{\cdot\}$.

<img src="./pngs/ldsv5a05_b1.png" width="250">

---

### c) Analytisches Signal mit `hilbert()`

In Python kann die Hilbert-Transformation wie folgt berechnet werden:

```python
from scipy.signal import hilbert

y = hilbert(x)
```

Der Rückgabewert `y` ist das analytische Signal zu `x`, der Realteil von `y` entspricht `x` und der Imaginärteil von `y` ist die Hilbert-Transformation von `x`.

Die folgende Abbildung zeigt das analytische Signal für die Sinusreihe aus Aufgabe a):

<img src="./pngs/ldsv5a05_c1.png" width="250">

Ein Vergleich mit den Ergebnissen aus Aufgabe b) zeigt, dass das Prinzip identisch ist, nur die Gruppendelay in den Abbildungen unterschiedlich ist.
